# 04 · Interpretación y simulación clínica

Entrada: `datos_cleveland_limpios.csv`, `modelos_tfm.joblib`.
Salida: odds ratios, importancia por permutación y simulación de tres pacientes.

In [ ]:
import os, sys, glob, shutil
import numpy as np
import pandas as pd
import joblib
import scipy
import sklearn
import statsmodels
import statsmodels.api as sm
import matplotlib
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

SEMILLA = 42
os.makedirs("figures", exist_ok=True)

try:
    from google.colab import files
    if not (glob.glob("datos_cleveland_limpios*.csv") and glob.glob("modelos_tfm*.joblib")):
        files.upload()
except ImportError:
    pass

df = pd.read_csv(sorted(glob.glob("datos_cleveland_limpios*.csv"))[0])
paquete = joblib.load(sorted(glob.glob("modelos_tfm*.joblib"))[0])
if sklearn.__version__ != paquete["sklearn"]:
    print(f"AVISO: modelos guardados con scikit-learn {paquete['sklearn']}, "
          f"entorno actual {sklearn.__version__}. Si hay errores, reejecutar el cuaderno 02.")
modelos, umbrales = paquete["modelos"], paquete["umbrales"]

X = df.drop(columns=["num", "enfermedad", "restecg"])
y = df["enfermedad"]
X_train, y_train = X.loc[paquete["indices_train"]], y.loc[paquete["indices_train"]]
X_test, y_test = X.loc[paquete["indices_test"]], y.loc[paquete["indices_test"]]
print("Entrenamiento:", X_train.shape, "| Test:", X_test.shape)

## 1. Odds ratios (logística sin penalizar, entrenamiento)

In [ ]:
def matriz_or(datos, modas):
    d = datos.fillna(modas)
    return pd.DataFrame({
        "Edad (por 10 años)": d["age"] / 10,
        "Sexo masculino": d["sex"],
        "Angina atípica (ref.: típica)": (d["cp"] == 2).astype(int),
        "Dolor no anginoso (ref.: típica)": (d["cp"] == 3).astype(int),
        "Asintomático (ref.: típica)": (d["cp"] == 4).astype(int),
        "PA sistólica (por 10 mmHg)": d["trestbps"] / 10,
        "Colesterol (por 10 mg/dl)": d["chol"] / 10,
        "Glucemia > 120 mg/dl": d["fbs"],
        "ECG en reposo anormal": d["restecg_anormal"],
        "FC máxima (por 10 lpm)": d["thalach"] / 10,
        "Angina con el ejercicio": d["exang"],
        "Descenso ST (por 1 mm)": d["oldpeak"],
        "Pendiente ST plana (ref.: ascendente)": (d["slope"] == 2).astype(int),
        "Pendiente ST descendente (ref.: ascendente)": (d["slope"] == 3).astype(int),
        "Vasos coloreados (por cada vaso)": d["ca"],
        "Talio: defecto fijo (ref.: normal)": (d["thal"] == 6).astype(int),
        "Talio: defecto reversible (ref.: normal)": (d["thal"] == 7).astype(int),
    })

modas = {"ca": X_train["ca"].mode()[0], "thal": X_train["thal"].mode()[0]}
M_train = matriz_or(X_train, modas)
print("Parámetros:", M_train.shape[1] + 1, "| eventos:", y_train.sum())

logit = sm.Logit(y_train, sm.add_constant(M_train)).fit(disp=False)
ic_or = np.exp(logit.conf_int())
tabla_or = pd.DataFrame({"OR": np.exp(logit.params), "IC 95 % inf": ic_or[0],
                         "IC 95 % sup": ic_or[1], "p": logit.pvalues}).drop(index="const")
tabla_or["p"] = tabla_or["p"].apply(lambda p: "< 0,001" if p < 0.001 else f"{p:.3f}".replace(".", ","))
tabla_or.to_csv("figures/tabla_odds_ratios.csv")
tabla_or.round(2)

In [ ]:
t = tabla_or.iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 7))
ax.errorbar(t["OR"], range(len(t)), xerr=[t["OR"] - t["IC 95 % inf"], t["IC 95 % sup"] - t["OR"]],
            fmt="o", color="#4C72B0", ecolor="grey", capsize=3)
ax.axvline(1, color="black", lw=0.8, ls="--")
ax.set_xscale("log")
ax.set_yticks(range(len(t)), t.index, fontsize=8)
ax.set_xlabel("Odds ratio (IC 95 %, escala logarítmica)")
ax.set_title(f"Asociación con enfermedad coronaria\n(logística sin penalizar, entrenamiento, n = {len(y_train)})", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_forest_odds_ratios.png", dpi=300)
plt.show()

## 2. Importancia por permutación (Random Forest completo, test)

In [ ]:
NOMBRES = {"age": "Edad", "sex": "Sexo", "cp": "Tipo de dolor", "trestbps": "PA sistólica",
           "chol": "Colesterol", "fbs": "Glucemia > 120", "restecg_anormal": "ECG en reposo",
           "thalach": "FC máxima", "exang": "Angina con ejercicio", "oldpeak": "Descenso ST",
           "slope": "Pendiente ST", "ca": "Vasos coloreados", "thal": "Talio"}

rf = modelos["Random Forest completo"]
res = permutation_importance(rf["modelo"], X_test[rf["variables"]], y_test,
                             scoring="roc_auc", n_repeats=50, random_state=SEMILLA)
imp = pd.DataFrame({"Caída media del AUC": res.importances_mean, "DE": res.importances_std},
                   index=[NOMBRES[v] for v in rf["variables"]]).sort_values("Caída media del AUC")
imp.to_csv("figures/tabla_importancia_permutacion.csv")

fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(imp.index, imp["Caída media del AUC"], xerr=imp["DE"], color="#55A868", ecolor="grey")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Caída del AUC al permutar la variable")
ax.set_title("Importancia por permutación · Random Forest completo (test)", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_importancia_permutacion.png", dpi=300)
plt.show()

## 3. Simulación clínica (logística completa)

Contribución de cada variable = coeficiente × (valor del paciente − valor del paciente medio de entrenamiento), en la escala transformada.

In [ ]:
final = modelos["Logística completa"]
pipe = final["modelo"]
prep, lr = pipe.named_steps["prep"], pipe.named_steps["modelo"]
coef = pd.Series(lr.coef_[0], index=prep.get_feature_names_out())
z_medio = pd.Series(prep.transform(X_train[final["variables"]]).mean(axis=0), index=coef.index)
UMBRAL = umbrales["Logística completa"]

ETIQUETAS = {"cp": {1.0: "angina típica", 2.0: "angina atípica", 3.0: "dolor no anginoso", 4.0: "asintomático"},
             "slope": {1.0: "pendiente ST ascendente", 2.0: "pendiente ST plana", 3.0: "pendiente ST descendente"},
             "thal": {3.0: "talio normal", 6.0: "talio: defecto fijo", 7.0: "talio: defecto reversible"}}

def nombre_legible(columna):
    grupo, var = columna.split("__", 1)
    if grupo == "nominales":
        var = var.rsplit("_", 1)[0]
    return NOMBRES[var]

def evaluar_paciente(paciente, n_motivos=3):
    datos = pd.DataFrame([paciente])[final["variables"]]
    p = pipe.predict_proba(datos)[0, 1]
    z = pd.Series(prep.transform(datos)[0], index=coef.index)
    contrib = (coef * (z - z_medio)).groupby(nombre_legible).sum().sort_values()
    valor = {NOMBRES[v]: f"{NOMBRES[v]} ({ETIQUETAS[v][float(paciente[v])]})" for v in ETIQUETAS}
    suben = [valor.get(n, n) for n in contrib[contrib > 0.05].index[::-1][:n_motivos]]
    bajan = [valor.get(n, n) for n in contrib[contrib < -0.05].index[:n_motivos]]
    print(f"Probabilidad estimada: {p:.0%}")
    print(f"  Umbral 0,5:            {'POSITIVO' if p >= 0.5 else 'negativo'}")
    print(f"  Umbral descarte ({UMBRAL:.2f}): "
          f"{'positivo: no descartar' if p >= UMBRAL else 'NEGATIVO: probabilidad baja'}")
    print("  Suben la probabilidad:", ", ".join(suben) or "ninguna destacable")
    print("  La bajan:             ", ", ".join(bajan) or "ninguna destacable")
    print("  Herramienta de apoyo; no sustituye el juicio clínico.")
    return p

In [ ]:
pacientes = {
    "A · Mujer de 45 años, dolor no anginoso, pruebas normales": dict(
        age=45, sex=0, cp=3, trestbps=125, chol=210, fbs=0, restecg_anormal=0,
        thalach=170, exang=0, oldpeak=0.0, slope=1, ca=0, thal=3),
    "B · Varón de 58 años, angina atípica, pruebas dudosas": dict(
        age=58, sex=1, cp=2, trestbps=140, chol=250, fbs=0, restecg_anormal=1,
        thalach=145, exang=0, oldpeak=1.0, slope=2, ca=1, thal=3),
    "C · Varón de 62 años, asintomático, ergometría y talio positivos": dict(
        age=62, sex=1, cp=4, trestbps=150, chol=280, fbs=1, restecg_anormal=1,
        thalach=120, exang=1, oldpeak=2.5, slope=2, ca=2, thal=7),
}
for titulo, datos_paciente in pacientes.items():
    print("\n" + titulo)
    evaluar_paciente(datos_paciente)

## 4. Versiones de software y exportación

In [ ]:
print("Python", sys.version.split()[0])
for lib in (pd, np, scipy, sklearn, statsmodels, matplotlib, joblib):
    print(f"{lib.__name__:<12} {lib.__version__}")

try:
    from google.colab import files
    shutil.make_archive("resultados_interpretacion", "zip", "figures")
    files.download("resultados_interpretacion.zip")
except ImportError:
    pass